# DeepSeek-v3.2 Evaluation
This Jupyter notebook implements an evaluation pipeline for the [DeepSeek-v3.2](https://api-docs.deepseek.com/) language model on Portuguese language sentiment analysis tasks. The notebook is designed to:



1.   Systematically evaluate the model's performance across multiple Brazilian Portuguese datasets (IMDB-PT, OpCovidBR, RELI, and TweetSentBR);
2.   Test different prompting strategies using various templates for few-shot and zero-shot learning;
3. Conduct multiple evaluation rounds (3 rounds);
4. Track inference time and model predictions;
5. Handle large-scale evaluations with checkpoint saving and recovery mechanisms.


The notebook uses the DeepSeek API via the `openai` library with async requests for efficient inference. Environment variables are loaded from a `.env` file via `python-dotenv`.


<br>

**Expected Environment Variables** (`.env` file)


*   `'AP_DATA_PREPROCESSED'`: Path to preprocessed data
*   `'AP_DATA_RESULTS'`: Path to results directory
*   `'AP_PROMPTS_TEMPLATE'`: Path to template files
*   `'AP_STAGIN-AREA_PATH'`: Path to staging area directory
*   `'DEEPSEEK_API_KEY'`: DeepSeek API Key


<br>

**Expected project structure**

```
└── Data
   ├── preprocessed
   │   ├── imdb-pt
   │   ├── opcovidbr
   │   ├── reli
   │   └── tweetsentbr
   ├── prompts_templates
   │   ├── template-task-agnostic-no-instructions-few-shot-en.jinja
   │   ├── template-task-specific-instructions-few-shot.jinja
   │   ├── template-task-specific-instructions-few-shot-json.jinja
   │   ├── template-task-specific-instructions-zero-shot.jinja
   │   ├── template-task-specific-meta-prompt-few-shot.jinja
   │   └── template-task-specific-roleplay-few-shot.jinja
   ├── raw
   │   ├── imdb-pt
   │   ├── opcovidbr
   │   ├── reli
   │   └── tweetsentbr
   ├── results
   │   ├── deepseek-v3.2
   │   ├── gemma2-9b-it
   │   ├── qwen3.5-9b
   │   ├── boto-9b-it
   │   └── multilingual-sentiment-analysis
   └── staging_area
```
<br>

**References**:
```
@misc{deepseek-v3,
  title={DeepSeek-V3 Technical Report},
  year={2024}
}
```

# Installations

Installing/updating packages.

In [ ]:
!pip install -U openai python-dotenv jinja2 tqdm numpy pandas

# 01 - Imports

Import the necessary libraries.



In [ ]:
import os
import glob
import asyncio
import numpy as np
import pandas as pd
import json
import jinja2
import tqdm

from dotenv import load_dotenv
from openai import AsyncOpenAI
from pathlib import Path
from time import time

load_dotenv()

# 02 - Constants

Definition of constants.

In [ ]:
MODEL_NAME = "deepseek-v3.2"

#Paths
PREPROCESSED_DATA_ROOT_PATH = Path(os.getenv('AP_DATA_PREPROCESSED'))
RESULTS_DATA_ROOT_PATH = Path(os.getenv('AP_DATA_RESULTS'))
TEMPLATE_DATA_ROOT_PATH = Path(os.getenv('AP_PROMPTS_TEMPLATE'))
STAGING_AREA_ROOT_PATH = Path(os.getenv('AP_STAGIN-AREA_PATH'))

#API Configuration
DEEPSEEK_API_KEY = os.getenv('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL = "https://api.deepseek.com"
MAX_CONCURRENT_REQUESTS = 30

client = AsyncOpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
)

#Data
TEMPLATES = {Path(item).stem: Path(item) for item in glob.glob(f'{TEMPLATE_DATA_ROOT_PATH}/*.jinja')}
TEST_DATASETS = {dataset.stem.split("_test")[0]: {"df": pd.read_csv(dataset), "path":dataset.parent} for dataset in PREPROCESSED_DATA_ROOT_PATH.glob('*/*_test.csv')}
DEMO_DATASETS = {dataset.stem.split("_demos")[0]: {"df": pd.read_csv(dataset), "path":dataset.parent} for dataset in PREPROCESSED_DATA_ROOT_PATH.glob('*/*_demos.csv')}
RESULT_DATASETS = [dataset.stem.split("_result")[0] for dataset in RESULTS_DATA_ROOT_PATH.glob(f'{MODEL_NAME}/*.csv')]


DATASETS_TASK_TYPES = {
    "imdb-pt": "binary-classification",
    "opcovidbr": "binary-classification",
    "reli": "multicass-classification",
    "tweetsentbr": "multicass-classification"
    }

TASK_LABELS = {"binary-classification":[(-1, "Negativo"), (1, "Positivo")],
          "multicass-classification":[(-1, "Negativo"),(0, "Neutro"), (1, "Positivo")]}

# 03 - Functions

Definition of the functions that will be used during predictions.

In [ ]:
def get_jinja_template(file_path: str | Path) -> tuple[str, jinja2.Template]:
    """
    Load and compile a Jinja2 template from a file.
    Args:
        file_path (str | Path): Path to the Jinja2 template file.
    Returns:
        tuple[str, jinja2.Template]: A tuple containing the template name and the compiled Jinja2 template object.
    """
    if not isinstance(file_path, Path):
        file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f'The file {file_path} does not exist.')

    template_name = file_path.stem

    with open(file_path, 'r') as template:
        template = jinja2.Template(template.read())
    return template_name, template

def get_demonstrations_df(dataset: pd.DataFrame, random_state: int = 42) -> pd.DataFrame:
    """
    Randomly shuffle a dataset for demonstration purposes.
    Args:
        dataset (pd.DataFrame): The input DataFrame to sample from.
        random_state (int, optional): Random seed for reproducibility. Defaults to 42.
    Returns:
        pd.DataFrame: The shuffled DataFrame values.
    """
    demo_df = dataset.sample(frac=1, random_state=random_state).values
    return demo_df

def generate_staging_area_csv_file_name(model_name: str, dataset_name: str, prompt_name: str, round: int, first_index: int, last_index: int) -> str:
    """
    Generate a standardized CSV file name for the staging area.
    Args:
        model_name (str): Name of the model.
        dataset_name (str): Name of the dataset.
        prompt_name (str): Name of the prompt template.
        round (int): Round number of the experiment.
        first_index (int): Starting index of the data range.
        last_index (int): Ending index of the data range.
    Returns:
        str: Formatted file name string following the pattern 'model_dataset_prompt_round_start-end.csv'.
    """
    file_name = f'{model_name}_{dataset_name}_{prompt_name}_{round}_{first_index}-{last_index}.csv'
    return file_name

def get_staging_area_files_by_model_or_dataset_name(model_name: str = None, dataset_name: str = None, prompt_name: str = None, round: int = None) -> list[Path]:
    """
    Retrieve sorted list of staging area files matching specified criteria.
    Args:
        model_name (str, optional): Model name to filter by. Defaults to None.
        dataset_name (str, optional): Dataset name to filter by. Defaults to None.
        prompt_name (str, optional): Prompt template name to filter by. Defaults to None.
        round (int, optional): Round number to filter by. Defaults to None.
    Returns:
        list[Path]: Sorted list of Path objects for matching CSV files.
    """
    files = [csv_file for csv_file in STAGING_AREA_ROOT_PATH.glob(f'{model_name}_{dataset_name}_{prompt_name}_{round}_*.csv')]
    return sorted(files, key=lambda x: int((str(x.absolute()).split('-')[-1]).split('.')[0]))

def get_last_staging_area_file_content_index(model_name: str = None, dataset_name: str = None, prompt_name: str = None, round: int = None) -> int | None:
    """
    Get the last index from staging area files for specified criteria.
    Args:
        model_name (str, optional): Model name to filter by. Defaults to None.
        dataset_name (str, optional): Dataset name to filter by. Defaults to None.
        prompt_name (str, optional): Prompt template name to filter by. Defaults to None.
        round (int, optional): Round number to filter by. Defaults to None.
    Returns:
        int: Last index found in matching files, or None if no files found.
    """
    files = get_staging_area_files_by_model_or_dataset_name(model_name=model_name, dataset_name=dataset_name, prompt_name=prompt_name, round=round)
    if len(files) == 0:
        return None
    last_file = str(files[-1].absolute())
    last_file_index_name = int((last_file.split('_')[-1]).split('.')[0].split('-')[-1])
    return last_file_index_name

def save_csv_to_staging_area(model_name: str, dataset_name: str, prompt_name: str, round: int, results_list: list) -> Path:
    """
    Save experiment results to a CSV file in the staging area.
    Args:
        model_name (str): Name of the model.
        dataset_name (str): Name of the dataset.
        prompt_name (str): Name of the prompt template.
        round (int): Round number of the experiment.
        results_list (list): List of tuples containing (index, prediction, time) results.
    Returns:
        Path: Path to the saved CSV file.
    """
    sorted_result_list = sorted(results_list, key=lambda x: x[0])
    first_index = sorted_result_list[0][0]
    last_index = sorted_result_list[-1][0]
    file_name = generate_staging_area_csv_file_name(
        model_name=model_name,
        dataset_name=dataset_name,
        prompt_name=prompt_name,
        round=round,
        first_index=first_index,
        last_index=last_index
        )

    file_path = STAGING_AREA_ROOT_PATH / file_name
    df = pd.DataFrame(results_list, columns=['index','predictions', 'time'])
    df.to_csv(file_path)
    print(f'File containing index {first_index} to {last_index} for  model {model_name},dataset {dataset_name}, template{prompt_name} and round{round} created at{file_path} with success.')
    return file_path.parent / (file_path.name +".csv")

def convert_csv_in_staging_area_to_dataframe(list_of_files: list[Path]) -> pd.DataFrame:
    """
    Combine multiple staging area CSV files into a single sorted DataFrame.
    Args:
        list_of_files (list[Path]): List of paths to CSV files in the staging area.
    Returns:
        pd.DataFrame: Combined and sorted DataFrame with predictions and timing information.
    """
    df = pd.DataFrame(columns=["Unmatch: 0", "index", "predictions", "time"])
    for file in list_of_files:
        temp_df = pd.read_csv(file)
        df = pd.concat([df, temp_df])
    df = df.sort_values(by='index')
    df = df.drop_duplicates(subset=['index'])
    df = df.reset_index(drop=True).set_index('index')
    df = df[['predictions', 'time']]
    return df

In [ ]:
TEST_DATASETS.keys()

## 04 - DeepSeek-v3.2 Evaluation

# 04.01 - Perform inference

Perform async inference via DeepSeek API for all datasets and prompt templates for 3 rounds.

In [ ]:
async def call_deepseek_api(semaphore, client, messages, template_name, index):
    """Make a single async API call to DeepSeek with semaphore-based rate limiting."""
    async with semaphore:
        t0 = time()
        max_tokens = 10 if "json" not in template_name else 20
        try:
            response = await client.chat.completions.create(
                model="deepseek-chat",
                messages=messages,
                max_tokens=max_tokens,
                temperature=0,
            )
            return (index, response.choices[0].message.content, time() - t0)
        except Exception as e:
            print(f"Error at index {index}: {e}")
            await asyncio.sleep(2)
            try:
                t0 = time()
                response = await client.chat.completions.create(
                    model="deepseek-chat",
                    messages=messages,
                    max_tokens=max_tokens,
                    temperature=0,
                )
                return (index, response.choices[0].message.content, time() - t0)
            except Exception as e2:
                print(f"Retry failed at index {index}: {e2}")
                return (index, f"ERROR: {e2}", time() - t0)


CHECKPOINT_INTERVAL = 100
semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

# Iterate through multiple rounds of evaluation (1-3)
for round in range(1, 4):
    # Process each test dataset
    for df_name, df in TEST_DATASETS.items():
         # Get task-specific labels based on dataset type
        task_labels = TASK_LABELS.get(DATASETS_TASK_TYPES.get(df_name))
        # Get demonstration dataset corresponding to current test dataset
        demo_df = DEMO_DATASETS.get(df_name)["df"]
        # Get shuffled demonstration examples
        examples = get_demonstrations_df(demo_df, random_state=42 if round==1 else round)

        # Process each template
        for template_path in TEMPLATES.values():
            template_name, template = get_jinja_template(template_path)

            # Check if this combination has already been evaluated
            if f"{MODEL_NAME}_{df_name}_{template_name}_{round}" not in RESULT_DATASETS:
                print(f'Starting {round} evaluation on dataset: {df_name} on template: {template_name}')
                results_list = []

                # Find the last processed index from previous runs (if any)
                first_index = get_last_staging_area_file_content_index(
                    dataset_name=df_name,
                    model_name=MODEL_NAME,
                    prompt_name=template_name,
                    round=round)

                # Start from beginning if no previous files found
                if first_index is None:
                    first_index = 0
                    print(f'No file were found for the current dataset and model, starting from index 0')
                # Otherwise, continue from last processed index
                else:
                    first_index += 1
                    print(f'One or more files were found. Starting from lastest index found ({first_index}) for dataset {df_name}')

                # Build list of items to process from first_index onwards
                items_to_process = list(df['df']['text'][first_index:].items())

                # Process in chunks for ordered checkpointing
                for chunk_start in range(0, len(items_to_process), CHECKPOINT_INTERVAL):
                    chunk = items_to_process[chunk_start:chunk_start + CHECKPOINT_INTERVAL]

                    # Create async tasks for this chunk
                    tasks = []
                    for index, item in chunk:
                        prompt = template.render(
                            examples=examples,
                            input_text=item,
                            labels=task_labels)

                        messages = [{"role": "user", "content": prompt}]
                        tasks.append(call_deepseek_api(semaphore, client, messages, template_name, index))

                    # Run chunk concurrently (semaphore limits actual concurrency)
                    chunk_results = await asyncio.gather(*tasks)
                    results_list.extend(chunk_results)

                    # Save checkpoint after each chunk
                    save_csv_to_staging_area(
                        model_name=MODEL_NAME,
                        dataset_name=df_name,
                        prompt_name=template_name,
                        round=round,
                        results_list=results_list)

                # Handle cases where results might be split across multiple files
                if len(results_list) != (df['df'].shape[0]):

                     # Combine all staging area files for this configuration
                    df_temp = convert_csv_in_staging_area_to_dataframe(
                        get_staging_area_files_by_model_or_dataset_name(
                            dataset_name=df_name,
                            model_name=MODEL_NAME,
                            prompt_name=template_name,
                            round=round))

                    # If combined results match dataset size, update the dataframe
                    if df_temp.shape[0] == df['df'].shape[0]:
                        df['df']['predictions'] = df_temp['predictions']
                        df['df']['time'] = df_temp["time"]

                # Handle case where all results are in single results_list
                elif len(results_list) == (df['df'].shape[0]):
                    df['df']['predictions'] = np.array(results_list, dtype=object)[:,1]
                    df['df']['time'] = np.array(results_list, dtype=object)[:,2]

                 # Raise error if results count doesn't match dataset size
                else:
                    raise ValueError('The number of results is different from the number of rows in the dataset.')

                 # Save final results to CSV
                df['df'].to_csv(f'{RESULTS_DATA_ROOT_PATH}/{MODEL_NAME}/{MODEL_NAME}_{df_name}_{template_name}_{round}_result.csv', index=False)

                 # Update list of completed evaluations
                RESULT_DATASETS = [dataset.stem.split("_result")[0] for dataset in RESULTS_DATA_ROOT_PATH.glob(f'{MODEL_NAME}/*.csv')]
                print(f'The evaluation of the round{round} on dataset: {df_name} on template: {template_name} has ended')